# Setup:

In [26]:
import torch
from transformers import DataCollatorWithPadding, RobertaConfig, RobertaModel, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, Trainer, TrainingArguments
from tqdm import tqdm
import os
from datasets import Dataset
import random
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
DATASET_ROOT = "../../CrossVul"
ALLOWED_CWE_IDS = {"CWE-22"} # "CWE-22", "CWE-89", "CWE-787"
LANGUAGES = ['c', 'cpp', 'cs', 'html', 'java', 'py', 'php']
SEED = 42

In [27]:
vulBERTa = "claudios/VulBERTa-MLP-ReVeal"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(vulBERTa, trust_remote_code=True)
print(device)

cuda


# Data Preprocessing

In [31]:
def collect_files_for_cwe(cwe_id):
    samples = []
    for lang in LANGUAGES:
        lang_dir = os.path.join(DATASET_ROOT, cwe_id, lang)
        if not os.path.isdir(lang_dir):
            continue
        for filename in os.listdir(lang_dir):
            filepath = os.path.join(lang_dir, filename)
            if filename.endswith('.DS_Store'):
                continue
            label = 1 if "bad" in filename.lower() else 0
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            samples.append({
                "filename": filename,
                "code": code,
                "label": label
            })
    print(len(samples))
    return samples

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

def tokenize_example(batch, cwe_id, max_length=512):
    #prompt = f"This code may contain CWE-{cwe_id}. Analyze carefully:\n"
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for code, label in zip(batch["code"], batch["label"]):
        #full_input = prompt + code
        tokens = tokenizer(code, return_attention_mask=True, truncation=False)
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]

        for i in range(0, len(input_ids), max_length):
            chunk_ids = input_ids[i:i + max_length]
            chunk_mask = attention_mask[i:i + max_length]

            pad_len = max_length - len(chunk_ids)
            if pad_len > 0:
                chunk_ids += [tokenizer.pad_token_id] * pad_len
                chunk_mask += [0] * pad_len

            input_ids_list.append(chunk_ids)
            attention_mask_list.append(chunk_mask)
            labels_list.append(label)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list
    }


# Model Finetuning:

In [ ]:
for cwe_id in ALLOWED_CWE_IDS:
    print(f"\n--- Training for {cwe_id} ---")
    samples = collect_files_for_cwe(cwe_id)
    
    random.seed(SEED)
    random.shuffle(samples)
    raw_dataset = Dataset.from_list(samples)
    tokenized_dataset = raw_dataset.map(tokenize_example, batched=True, remove_columns=["filename", "code"], fn_kwargs={"cwe_id": cwe_id})
    train_test = tokenized_dataset.train_test_split(test_size=0.2, seed=SEED)
    train_dataset = train_test["train"]
    eval_dataset = train_test["test"]

    model_path = f"./models/vulberta_{cwe_id}/final"

    print(f"Training new model for {cwe_id}...")
    model = AutoModelForSequenceClassification.from_pretrained(vulBERTa, num_labels=2).to(device)

    training_args = TrainingArguments(
        output_dir=f"./models/vulberta_{cwe_id}",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model(model_path)


--- Training for CWE-22 ---
320


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

4156
Training new model for CWE-22...


C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.695888,0.461538,0.475676,0.854369,0.611111
2,0.714200,0.713672,0.509615,1.000000,0.009709,0.019231
3,0.694100,0.719702,0.454327,0.471311,0.837379,0.603147
4,0.688300,0.761886,0.370192,0.383817,0.449029,0.413870
5,0.659300,0.917910,0.375000,0.370813,0.376214,0.373494


# Model Evaluation

In [ ]:
def predict_with_chunk_voting(trainer, raw_samples, chunk_size=512, stride=256):
    true_labels = []
    pred_labels = []

    for example in tqdm(raw_samples, desc="Evaluating with chunk voting"):
        label = example["label"]
        true_labels.append(label)

        tokens = tokenizer(example["code"], return_attention_mask=True, truncation=False)
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]

        chunks = []
        for i in range(0, len(input_ids), stride):
            chunk_ids = input_ids[i:i + chunk_size]
            chunk_mask = attention_mask[i:i + chunk_size]

            chunks.append({
                "input_ids": chunk_ids,
                "attention_mask": chunk_mask,
            })

        if not chunks:
            pred_labels.append(0)
            continue

        max_len = max(len(c["input_ids"]) for c in chunks)
        for chunk in chunks:
            pad_len = max_len - len(chunk["input_ids"])
            chunk["input_ids"] += [tokenizer.pad_token_id] * pad_len
            chunk["attention_mask"] += [0] * pad_len

        input_ids = torch.tensor([c["input_ids"] for c in chunks]).to(trainer.model.device)
        attention_mask = torch.tensor([c["attention_mask"] for c in chunks]).to(trainer.model.device)

        with torch.no_grad():
            outputs = trainer.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

        file_pred = 1 if (preds.mean() > 0.2) else 0
        pred_labels.append(file_pred)

    return true_labels, pred_labels

true_labels, pred_labels = predict_with_chunk_voting(trainer, samples) 
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division=0)
acc = accuracy_score(true_labels, pred_labels)

print(f"Metrics for {cwe_id}:")
print({
    'accuracy': acc,
    'precision': precision,
    'recall': recall,
    'f1': f1,
})

print(f"\nConfusion Matrix for {cwe_id}:")
print(confusion_matrix(true_labels, pred_labels))

Evaluating with chunk voting:   2%|▏         | 44/2142 [00:33<26:34,  1.32it/s]


KeyboardInterrupt: 